# 0.0 Imports

In [4]:
import pandas as pd
import numpy as np

# 0.1 Funções

# 0.2 Loading DataSet

In [5]:
df_raw = pd.read_excel('dataset/dataset_leite_fazenda_100_animais_900_dias.xlsx')

/home/lucas/.pyenv/versions/milk_env/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/home/lucas/.pyenv/versions/milk_env/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [20]:
#verificação shape do dataset
print(df_raw.shape)

#primeira cópia df_raw
df = df_raw.copy()

df.head(10).sort_values(by=['Date'], ascending=True).T

(6100, 49)


,0,1,2,3,4,5,6,7,8,9
Farm_ID,FARM_001,FARM_001,FARM_001,FARM_001,FARM_001,FARM_001,FARM_001,FARM_001,FARM_001,FARM_001
Cattle_ID,CATTLE_001,CATTLE_001,CATTLE_001,CATTLE_001,CATTLE_001,CATTLE_001,CATTLE_001,CATTLE_001,CATTLE_001,CATTLE_001
Date,2024-01-01 00:00:00,2024-01-16 00:00:00,2024-01-31 00:00:00,2024-02-15 00:00:00,2024-03-01 00:00:00,2024-03-16 00:00:00,2024-03-31 00:00:00,2024-04-15 00:00:00,2024-04-30 00:00:00,2024-05-15 00:00:00
Animal_Category,Cow,Cow,Cow,Cow,Cow,Cow,Cow,Cow,Cow,Cow
Breed,Holstein,Holstein,Holstein,Holstein,Holstein,Holstein,Holstein,Holstein,Holstein,Holstein
Country,BR,BR,BR,BR,BR,BR,BR,BR,BR,BR
State,MG,MG,MG,MG,MG,MG,MG,MG,MG,MG
Region,Southeast,Southeast,Southeast,Southeast,Southeast,Southeast,Southeast,Southeast,Southeast,Southeast
Municipality,Vicosa,Vicosa,Vicosa,Vicosa,Vicosa,Vicosa,Vicosa,Vicosa,Vicosa,Vicosa
Climate_Zone,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude,Tropical_Altitude


# 0.3 Baseline do Negócio

Quanto de CMS (Consumo de Matéria Seca) vai ser necessário para fornecer ao rebanho da fazenda nos próximos 180 dias.

boa previsão do leite -> boa previsão de consumo de MS -> investimento acertivo em volumoso -> economia de até 20% sobre o total da receita

CMS = (0,0185xPeso do animal) + (0,305xLitros do animal produzido)

In [ ]:
# =================================================================
# Simulando da forma mais simples: CMS = peso_vivo * 0.03% parte 1
# =================================================================

"""
    Cada linha é uma coleta de dados no campo. Cada coleta são de 15 em 15 dias. 
    180 dias equivale a 12 observações em campo, ou 12 linhas do dataset.
"""

print('='*60)

#numero de animais
numero_de_vacas = df['Cattle_ID'].nunique()
print(f'Número de animais: {numero_de_vacas} vacas')

print('='*60)

#peso vivo dos animais por dia
peso_vivo_total_diario = df.groupby('Date')['Weight_kg'].sum()
print(f'Peso vivo total dos animais: {peso_vivo_total_diario:}kg')

print('='*60)

#rolling pega a linha 0 até a linha 11, ou seja, 12 datas/coletas
#peso vivo acumulado dos animais nos proximos 180 dias
peso_vivo_180d_total = peso_vivo_total_diario.rolling(12, min_periods=1).sum()
print(f'Peso vivo total acumulado dos animais dos proximos 180 dias somados: {peso_vivo_180d_total:}kg')

print('='*60)
#CMS de matéria seca diária (3% do PV)
CMS = df.groupby('Date')['Weight_kg'].sum()*0.03
print(f'Consumo de Matérica seca por peso vivo por dia: {CMS}kg')

print('='*60)

#Consumo de matéria seca acumulada nos proximos 180 dias
CMS_180d_total = CMS.rolling(12, min_periods=1).sum()
print(f'CMS acumulado dos proximos 180 dias somados: {CMS_180d_total:}kg')

print('='*60)

Número de animais: 100 vacas
Peso vivo total dos animais: Date
2024-01-01    59520.9
2024-01-16    59732.4
2024-01-31    59799.7
2024-02-15    59874.5
2024-03-01    59988.5
               ...   
2026-04-20    63254.3
2026-05-05    63192.5
2026-05-20    63150.9
2026-06-04    63277.7
2026-06-19    63212.8
Name: Weight_kg, Length: 61, dtype: float64kg
Peso vivo total acumulado dos animais dos proximos 180 dias somados: Date
2024-01-01     59520.9
2024-01-16    119253.3
2024-01-31    179053.0
2024-02-15    238927.5
2024-03-01    298916.0
                ...   
2026-04-20    759360.6
2026-05-05    759324.6
2026-05-20    759160.4
2026-06-04    759170.3
2026-06-19    759022.3
Name: Weight_kg, Length: 61, dtype: float64kg
Consumo de Matérica seca por peso vivo por dia: Date
2024-01-01    1785.627
2024-01-16    1791.972
2024-01-31    1793.991
2024-02-15    1796.235
2024-03-01    1799.655
                ...   
2026-04-20    1897.629
2026-05-05    1895.775
2026-05-20    1894.527
2026-06-04    18

In [27]:
# =================================================================
# Simulando da forma mais simples: CMS = peso_vivo * 0.03% parte 2
# =================================================================

#consumo de materia natural silagem proximos 180 dias (kg)
CMN_180d = CMS_180d_total / 0.3

#custo de materia natural silagem proximos 180 dias (R$/kg = 0,20 ) (R$)
custo_CMN_180d = CMN_180d * 0.2

#custo acumulado de materia natural silagem proximos 180 dias
custo_CMN_180d.sum()
print(f'custo com matéria natural nos próximos 180 dias é de R${custo_CMN_180d.sum():.2f}')

custo com matéria natural nos próximos 180 dias é de R$826651.44
